In [1]:
import os

In [2]:
# !rm -rf /kaggle/working/multi-model-fact-checking
!git clone https://github.com/tonyvu1289/multi-model-fact-checking.git

Cloning into 'multi-model-fact-checking'...
remote: Enumerating objects: 202, done.
remote: Counting objects: 100% (202/202), done.
remote: Compressing objects: 100% (118/118), done.
remote: Total 202 (delta 96), reused 183 (delta 77), pack-reused 0 (from 0)
Receiving objects: 100% (202/202), 98.45 KiB | 5.47 MiB/s, done.
Resolving deltas: 100% (96/96), done.


In [3]:
os.chdir('/kaggle/working/multi-model-fact-checking/')

In [4]:
!git fetch origin ocr-kaggle-ready

From https://github.com/tonyvu1289/multi-model-fact-checking
 * branch            ocr-kaggle-ready -> FETCH_HEAD


In [5]:
!git checkout ocr-kaggle-ready

Branch 'ocr-kaggle-ready' set up to track remote branch 'ocr-kaggle-ready' from 'origin'.
Switched to a new branch 'ocr-kaggle-ready'


In [6]:
!chmod +x task2/setup_ocr_env.sh task2/train.sh
!PYTHON_BIN=python bash task2/setup_ocr_env.sh
!python task2/precompute_ocr_cache.py --path /kaggle/input/mocheg1/mocheg --split all --output task2/ocr_cache_mocheg.json --vision_pt ocr_easyocr

[setup] Upgrading pip tooling...
[setup] Installing OCR training dependencies...
[setup] EasyOCR import check: OK
[setup] Pre-downloading EasyOCR model weights...
Using CPU. Note: This module is much faster with a GPU.
[setup] EasyOCR warmup done
[setup] Verifying Task2 train entrypoint imports...
[setup] Import check passed: MultiModalClassification
[setup] Completed successfully.
100%|█████████████████████████████████████| 2442/2442 [00:01<00:00, 1830.69it/s]
Total unique image evidences: 113017
  4%|█▍                                | 4593/113017 [31:30<12:23:55,  2.43it/s]
Traceback (most recent call last):
  File "/kaggle/working/multi-model-fact-checking/task2/precompute_ocr_cache.py", line 80, in <module>
    main()
  File "/kaggle/working/multi-model-fact-checking/task2/precompute_ocr_cache.py", line 61, in main
    text = ocr_encoder.extract_text(image_path)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/multi-model-fact-checking/task2/model.py", line 

In [ ]:
# Reuse cache from imported Kaggle dataset if available, else fallback to local precomputed cache.
# Replace <dataset-slug-folder> with the dataset folder name under /kaggle/input.
import os

EXTERNAL_CACHE = "/kaggle/input/<dataset-slug-folder>/ocr_cache_mocheg.json"
LOCAL_CACHE = "/kaggle/working/multi-model-fact-checking/task2/ocr_cache_mocheg.json"
OCR_CACHE_PATH = EXTERNAL_CACHE if os.path.exists(EXTERNAL_CACHE) else LOCAL_CACHE
print("Using OCR cache:", OCR_CACHE_PATH)

In [ ]:
# Optional: publish OCR cache as a Kaggle Dataset version for reuse across sessions.
# 1) Set these once for your account/dataset slug.
KAGGLE_DATASET_SLUG = "your-username/mocheg-ocr-cache"
CACHE_FILE = "task2/ocr_cache_mocheg.json"

# 2) Create dataset folder + metadata.
!mkdir -p /kaggle/working/ocr_cache_dataset
!cp {CACHE_FILE} /kaggle/working/ocr_cache_dataset/ocr_cache_mocheg.json

import json
meta = {
    "title": "mocheg-ocr-cache",
    "id": KAGGLE_DATASET_SLUG,
    "licenses": [{"name": "CC0-1.0"}]
}
with open('/kaggle/working/ocr_cache_dataset/dataset-metadata.json', 'w') as f:
    json.dump(meta, f)

# 3) Create first version if not exists, otherwise create a new version.
!kaggle datasets version -p /kaggle/working/ocr_cache_dataset -m "update OCR cache" -r zip

In [ ]:
!DATA_PATH=/kaggle/input/mocheg1/mocheg OCR_CACHE_PATH=$OCR_CACHE_PATH BATCH_SIZE=64 EPOCH=30 VISION_PT=ocr_easyocr bash task2/train.sh

100%|█████████████████████████████████████| 2442/2442 [00:01<00:00, 1778.61it/s]
MCVE model
Loading weights: 100%|█| 197/197 [00:00<00:00, 1737.96it/s, Materializing param=
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
roberta-base
Loading weights: 100%|█| 271/271 [00:00<00